# 02 — Data Cleaning & ETL Pipeline
**Project:** Brazilian E-Commerce Delivery & Satisfaction Analysis  
**Dataset:** Olist Brazilian E-Commerce (Kaggle)  
**Author:** Member 1 (ETL Lead)  
**Objective:** Build a fully documented, reproducible cleaning pipeline. Output a single master DataFrame to `data/processed/olist_cleaned.csv` ready for EDA and statistical analysis.

---
### Cleaning Steps Overview
1. Load raw files  
2. Parse & engineer datetime columns  
3. Compute delivery delay  
4. Handle missing values  
5. Standardize categorical columns  
6. Merge into master DataFrame  
7. Final validation  
8. Export to processed/

## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

RAW_PATH       = '../data/raw/'
PROCESSED_PATH = '../data/processed/'

os.makedirs(PROCESSED_PATH, exist_ok=True)

print('Configuration set. Paths ready.')

Configuration set. Paths ready.


## 2. Load Raw Files

In [2]:
# Load all 6 raw CSVs — never modify files in data/raw/
orders      = pd.read_csv(RAW_PATH + 'olist_orders_dataset.csv')
order_items = pd.read_csv(RAW_PATH + 'olist_order_items_dataset.csv')
customers   = pd.read_csv(RAW_PATH + 'olist_customers_dataset.csv')
reviews     = pd.read_csv(RAW_PATH + 'olist_order_reviews_dataset.csv')
sellers     = pd.read_csv(RAW_PATH + 'olist_sellers_dataset.csv')
products    = pd.read_csv(RAW_PATH + 'olist_products_dataset.csv')

print(f'orders:      {orders.shape}')
print(f'order_items: {order_items.shape}')
print(f'customers:   {customers.shape}')
print(f'reviews:     {reviews.shape}')
print(f'sellers:     {sellers.shape}')
print(f'products:    {products.shape}')

orders:      (99441, 8)
order_items: (112650, 7)
customers:   (99441, 5)
reviews:     (99224, 7)
sellers:     (3095, 4)
products:    (32951, 9)


## 3. Parse Datetime Columns

In [3]:
# TRANSFORMATION 01: Convert all date strings to datetime objects
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')
    print(f'  Parsed: {col}')

print(f'\nDatetime columns parsed. Null count after parsing:')
print(orders[datetime_cols].isnull().sum().to_string())

  Parsed: order_purchase_timestamp
  Parsed: order_approved_at
  Parsed: order_delivered_carrier_date
  Parsed: order_delivered_customer_date
  Parsed: order_estimated_delivery_date

Datetime columns parsed. Null count after parsing:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0


## 4. Compute Derived Columns

In [4]:
# TRANSFORMATION 02: Delivery delay in days (positive = late, negative = early)
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']
).dt.days

# TRANSFORMATION 03: Actual delivery time from purchase to delivery
orders['actual_delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

# TRANSFORMATION 04: Order purchase month and year for time-series analysis
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
orders['purchase_year']  = orders['order_purchase_timestamp'].dt.year
orders['purchase_dow']   = orders['order_purchase_timestamp'].dt.day_name()  # day of week

print('Derived columns created:')
print('  delivery_delay_days  — positive = late, negative = early')
print('  actual_delivery_days — total days from purchase to delivery')
print('  purchase_month       — YYYY-MM period string')
print('  purchase_year        — integer year')
print('  purchase_dow         — day of week name')
print(f'\nDelay stats:')
print(orders['delivery_delay_days'].describe())

Derived columns created:
  delivery_delay_days  — positive = late, negative = early
  actual_delivery_days — total days from purchase to delivery
  purchase_month       — YYYY-MM period string
  purchase_year        — integer year
  purchase_dow         — day of week name

Delay stats:
count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64


## 5. Handle Missing Values

In [5]:
# TRANSFORMATION 05: Drop orders with no delivered date that are marked as 'delivered'
# These are data quality errors — delivered status with no delivery timestamp
before = len(orders)
delivered_mask = orders['order_status'] == 'delivered'
orders = orders[~(delivered_mask & orders['order_delivered_customer_date'].isnull())]
after = len(orders)
print(f'Dropped {before - after} rows: delivered status but missing delivery date')

# TRANSFORMATION 06: Fill missing review comments with empty string (not analytical columns)
reviews['review_comment_title']   = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')
print(f'Review comment nulls filled with empty string')

# TRANSFORMATION 07: Fill missing product category with 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')
print(f'Product category nulls filled with unknown')

# TRANSFORMATION 08: Fill missing product dimensions with median (not critical for our analysis)
dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in dim_cols:
    if col in products.columns:
        median_val = products[col].median()
        products[col] = products[col].fillna(median_val)
        print(f'  {col}: filled nulls with median = {median_val:.1f}')

Dropped 8 rows: delivered status but missing delivery date
Review comment nulls filled with empty string
Product category nulls filled with unknown
  product_weight_g: filled nulls with median = 700.0
  product_length_cm: filled nulls with median = 25.0
  product_height_cm: filled nulls with median = 13.0
  product_width_cm: filled nulls with median = 20.0


## 6. Standardize Categorical Columns

In [6]:
# TRANSFORMATION 09: Standardize order_status to lowercase stripped strings
orders['order_status'] = orders['order_status'].str.strip().str.lower()
print('Order status values after standardization:')
print(orders['order_status'].value_counts().to_string())

# TRANSFORMATION 10: Standardize product_category_name — replace underscores, title case
products['product_category_name'] = (
    products['product_category_name']
    .str.replace('_', ' ', regex=False)
    .str.strip()
    .str.lower()
)
print(f'\nTop 10 product categories after standardization:')
print(products['product_category_name'].value_counts().head(10).to_string())

# TRANSFORMATION 11: Classify delivery as on-time or late
orders['delivery_status'] = np.where(
    orders['delivery_delay_days'].isnull(), 'unknown',
    np.where(orders['delivery_delay_days'] <= 0, 'on_time', 'late')
)
print(f'\nDelivery status distribution:')
print(orders['delivery_status'].value_counts().to_string())

Order status values after standardization:
order_status
delivered      96470
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

Top 10 product categories after standardization:
product_category_name
cama mesa banho           3029
esporte lazer             2867
moveis decoracao          2657
beleza saude              2444
utilidades domesticas     2335
automotivo                1900
informatica acessorios    1639
brinquedos                1411
relogios presentes        1329
telefonia                 1134

Delivery status distribution:
delivery_status
on_time    89941
late        6535
unknown     2957


## 7. Aggregate Order Items

In [7]:
# TRANSFORMATION 12: Aggregate order_items to order level
# Multiple items per order — collapse to one row per order_id
items_agg = order_items.groupby('order_id').agg(
    total_items    = ('order_item_id', 'count'),
    total_price    = ('price', 'sum'),
    total_freight  = ('freight_value', 'sum'),
    avg_item_price = ('price', 'mean')
).reset_index()

# Total order value including freight
items_agg['total_order_value'] = items_agg['total_price'] + items_agg['total_freight']

print(f'Order items aggregated: {len(items_agg):,} unique orders')
print(items_agg.describe())

Order items aggregated: 98,666 unique orders
        total_items   total_price  total_freight  avg_item_price  \
count  98666.000000  98666.000000   98666.000000    98666.000000   
mean       1.141731    137.754076      22.823562      125.919255   
std        0.538452    210.645145      21.650909      190.985636   
min        1.000000      0.850000       0.000000        0.850000   
25%        1.000000     45.900000      13.850000       41.990000   
50%        1.000000     86.900000      17.170000       79.000000   
75%        1.000000    149.900000      24.040000      139.900000   
max       21.000000  13440.000000    1794.960000     6735.000000   

       total_order_value  
count       98666.000000  
mean          160.577638  
std           220.466087  
min             9.590000  
25%            61.980000  
50%           105.290000  
75%           176.870000  
max         13664.080000  


## 8. Aggregate Reviews

In [8]:
# TRANSFORMATION 13: Some orders have multiple reviews — keep the latest one
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')

reviews_dedup = (
    reviews.sort_values('review_creation_date', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
[['order_id', 'review_score', 'review_creation_date']]
)

print(f'Reviews before dedup: {len(reviews):,}')
print(f'Reviews after dedup:  {len(reviews_dedup):,}')
print(f'\nReview score distribution:')
print(reviews_dedup['review_score'].value_counts().sort_index().to_string())

Reviews before dedup: 99,224
Reviews after dedup:  98,673

Review score distribution:
review_score
1    11364
2     3130
3     8133
4    19044
5    57002


## 9. Build Master DataFrame

In [9]:
# TRANSFORMATION 14: Merge all tables into one master DataFrame

# Start with orders (core fact table)
master = orders.copy()

# Join customers (many orders → one customer)
master = master.merge(customers[['customer_id', 'customer_state', 'customer_city']],
                      on='customer_id', how='left')
print(f'After joining customers:   {master.shape}')

# Join aggregated order items
master = master.merge(items_agg, on='order_id', how='left')
print(f'After joining items_agg:   {master.shape}')

# Join reviews
master = master.merge(reviews_dedup, on='order_id', how='left')
print(f'After joining reviews:     {master.shape}')

print(f'\nMaster DataFrame shape: {master.shape}')
print(f'Columns: {list(master.columns)}')

After joining customers:   (99433, 16)
After joining items_agg:   (99433, 21)
After joining reviews:     (99433, 23)

Master DataFrame shape: (99433, 23)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days', 'actual_delivery_days', 'purchase_month', 'purchase_year', 'purchase_dow', 'delivery_status', 'customer_state', 'customer_city', 'total_items', 'total_price', 'total_freight', 'avg_item_price', 'total_order_value', 'review_score', 'review_creation_date']


## 10. Filter to Delivered Orders Only

In [10]:
# TRANSFORMATION 15: For delivery analysis, keep only delivered orders
# Other statuses (cancelled, unavailable) cannot contribute to delivery metrics
before = len(master)
master_delivered = master[master['order_status'] == 'delivered'].copy()
after = len(master_delivered)

print(f'Total orders:     {before:,}')
print(f'Delivered orders: {after:,}')
print(f'Dropped:          {before - after:,} ({(before-after)/before*100:.1f}%)')
print(f'\nFinal dataset for analysis: {master_delivered.shape}')

Total orders:     99,433
Delivered orders: 96,470
Dropped:          2,963 (3.0%)

Final dataset for analysis: (96470, 23)


## 11. Final Validation

In [11]:
print('=== FINAL VALIDATION ===')
print(f'\nShape: {master_delivered.shape}')
print(f'Rows meet minimum requirement (5,000+): {len(master_delivered) >= 5000}')
print(f'Columns meet minimum requirement (8+): {master_delivered.shape[1] >= 8}')

print(f'\nNull summary (analytical columns only):')
key_cols = [
    'order_id', 'customer_id', 'order_status',
    'delivery_delay_days', 'actual_delivery_days',
    'delivery_status', 'review_score',
    'total_order_value', 'customer_state', 'purchase_month'
]
print(master_delivered[key_cols].isnull().sum().to_string())

print(f'\nDuplicate order_ids: {master_delivered["order_id"].duplicated().sum()}')

print(f'\nSample of final master DataFrame:')
display(master_delivered[key_cols].head(5))

=== FINAL VALIDATION ===

Shape: (96470, 23)
Rows meet minimum requirement (5,000+): True
Columns meet minimum requirement (8+): True

Null summary (analytical columns only):
order_id                  0
customer_id               0
order_status              0
delivery_delay_days       0
actual_delivery_days      0
delivery_status           0
review_score            646
total_order_value         0
customer_state            0
purchase_month            0

Duplicate order_ids: 0

Sample of final master DataFrame:


,order_id,customer_id,order_status,delivery_delay_days,actual_delivery_days,delivery_status,review_score,total_order_value,customer_state,purchase_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,-8.0,8.0,on_time,4.0,38.71,SP,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,-6.0,13.0,on_time,4.0,141.46,BA,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,-18.0,9.0,on_time,5.0,179.12,GO,2018-08
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,-13.0,13.0,on_time,5.0,72.20,RN,2017-11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,-10.0,2.0,on_time,5.0,28.62,SP,2018-02


## 12. Export to Processed

In [12]:
# Export cleaned master dataset
output_path = PROCESSED_PATH + 'olist_cleaned.csv'
master_delivered.to_csv(output_path, index=False)

# Verify file written correctly
verification = pd.read_csv(output_path)
print(f'Exported: {output_path}')
print(f'Verified shape: {verification.shape}')
print(f'\nCleaning pipeline complete. Team can now use olist_cleaned.csv for:')
print('  M2 → 03_eda.ipynb')
print('  M3 → 04_statistical_analysis.ipynb')
print('  M4 → 05_final_load_prep.ipynb + Tableau')

Exported: ../data/processed/olist_cleaned.csv
Verified shape: (96470, 23)

Cleaning pipeline complete. Team can now use olist_cleaned.csv for:
  M2 → 03_eda.ipynb
  M3 → 04_statistical_analysis.ipynb
  M4 → 05_final_load_prep.ipynb + Tableau


## 13. Cleaning Log Summary

| # | Transformation | Reason |
|---|---|---|
| 01 | Parsed 5 datetime columns | Enable time arithmetic |
| 02 | Computed `delivery_delay_days` | Core KPI for analysis |
| 03 | Computed `actual_delivery_days` | Logistics performance metric |
| 04 | Extracted `purchase_month`, `purchase_year`, `purchase_dow` | Time-series grouping |
| 05 | Dropped delivered orders with null delivery date | Data quality error rows |
| 06 | Filled review comment nulls with empty string | Non-analytical text column |
| 07 | Filled product category nulls with 'unknown' | Preserve row, flag missing |
| 08 | Filled product dimension nulls with median | Non-critical, preserve row |
| 09 | Standardized `order_status` to lowercase | Consistent categorical values |
| 10 | Standardized `product_category_name` | Remove underscores, uniform case |
| 11 | Created `delivery_status` (on_time / late) | Binary classification column |
| 12 | Aggregated `order_items` to order level | One row per order |
| 13 | Deduplicated reviews (keep latest) | Prevent duplicate join inflation |
| 14 | Merged all tables into master DataFrame | Single analysis-ready dataset |
| 15 | Filtered to delivered orders only | Focus analysis on completed orders |